# 23 — Financial Forecasting & Budget Optimization

**Why this matters at ServiceTitan**: the **Lead Data Scientist (Operations,
supporting Finance)** req lists "time series forecasting, anomaly detection,
optimization" as core requirements with 5+ years experience demanded. The job
directly works on financial planning, budget allocation, and FP&A decisions
that get presented to executives.

This notebook covers the four primitives that show up in any FP&A data-science
role:

| Section | Topic | Library |
|---------|-------|---------|
| 1 | Hierarchical revenue forecasting | sklearn + manual reconciliation |
| 2 | Probabilistic / quantile forecasting | sklearn GBM with quantile loss |
| 3 | Anomaly detection in financial metrics | rolling z-score + Isolation Forest |
| 4 | Resource-constrained budget optimization | scipy.optimize.linprog |
| 5 | Monte Carlo scenario simulation | numpy.random |
| 6 | Wiring forecasts → optimization | end-to-end pipeline |

All synthetic data, all runs in seconds, all code is production-shaped.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, IsolationForest
from sklearn.linear_model import QuantileRegressor
from scipy.optimize import linprog, minimize
import matplotlib
matplotlib.use("Agg")   # headless
import matplotlib.pyplot as plt

np.random.seed(0)


---
## Section 1 — Hierarchical Revenue Forecasting

### The problem
At ST, revenue rolls up across a hierarchy:

```
Total Revenue
├── Residential
│   ├── HVAC
│   ├── Plumbing
│   └── Electrical
└── Commercial
    ├── Mechanical
    └── Refrigeration
```

Finance needs forecasts at **every level** — leaf, segment, and total. The catch:
**bottom-up forecasts almost never reconcile** with top-down forecasts. A
finance-grade pipeline produces consistent numbers across the hierarchy.

### Three reconciliation strategies

| Strategy | What it does | Pro | Con |
|----------|--------------|-----|-----|
| **Bottom-up** | Forecast each leaf, sum up | Captures detail | Errors compound |
| **Top-down** | Forecast total, allocate by historical share | Smooth at top | Misses leaf trends |
| **MinT (chosen)** | Reconcile via optimal linear combo | Mathematically optimal | More complex |

For brevity we'll implement bottom-up with **proportional reconciliation** —
forecast each leaf with a GBM, then scale to match a top-level forecast. This
is the workhorse approach at most SaaS companies.


In [ ]:
# ── Generate synthetic hierarchical revenue data ──────────────────────────
# Monthly data, 3 years (36 points), 5 leaf segments
months = pd.date_range("2023-01-01", periods=36, freq="MS")
np.random.seed(42)

# Each leaf has different baseline, growth, seasonality, and noise
leaves = ["res_hvac", "res_plumb", "res_elec", "com_mech", "com_refrig"]
params = {
    "res_hvac":   {"base": 5_000_000, "growth": 0.015, "season_amp": 0.30, "noise": 0.05},  # peaks in summer
    "res_plumb":  {"base": 2_000_000, "growth": 0.010, "season_amp": 0.10, "noise": 0.07},
    "res_elec":   {"base": 1_500_000, "growth": 0.020, "season_amp": 0.08, "noise": 0.06},
    "com_mech":   {"base": 3_000_000, "growth": 0.025, "season_amp": 0.15, "noise": 0.08},
    "com_refrig": {"base": 1_200_000, "growth": 0.018, "season_amp": 0.20, "noise": 0.10},
}

def gen_series(p):
    t = np.arange(36)
    trend = p["base"] * (1 + p["growth"]) ** t
    season = 1 + p["season_amp"] * np.sin(2 * np.pi * (t - 6) / 12)  # summer peak
    noise = np.random.normal(1, p["noise"], 36)
    return trend * season * noise

revenue = pd.DataFrame({l: gen_series(p) for l, p in params.items()}, index=months)
revenue["total"] = revenue.sum(axis=1)
revenue["residential"] = revenue[["res_hvac", "res_plumb", "res_elec"]].sum(axis=1)
revenue["commercial"]  = revenue[["com_mech", "com_refrig"]].sum(axis=1)

print(revenue.tail(3).round(0))
print(f"\nShape: {revenue.shape}, columns: {list(revenue.columns)}")


In [ ]:
# ── Bottom-up forecaster with GBM + proportional reconciliation ──────────
def make_features(series: pd.Series, max_lag: int = 12) -> pd.DataFrame:
    """Lag and seasonal features."""
    df = pd.DataFrame({"y": series.values}, index=series.index)
    for lag in [1, 2, 3, 6, 12]:
        df[f"lag_{lag}"] = df["y"].shift(lag)
    df["month"] = df.index.month
    df["t"] = np.arange(len(df))
    return df


def forecast_leaf(series: pd.Series, horizon: int = 6) -> np.ndarray:
    """One-step-ahead GBM forecast, rolled forward horizon steps."""
    df = make_features(series).dropna()
    X, y = df.drop(columns="y"), df["y"]
    model = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=0)
    model.fit(X, y)

    history = list(series.values)
    forecasts = []
    for h in range(horizon):
        # build features for the next time step
        last_idx = len(history) - 1
        future_t = last_idx + 1
        future_month = ((series.index[-1].month + h) % 12) + 1
        feat = {
            "lag_1":  history[-1],
            "lag_2":  history[-2],
            "lag_3":  history[-3],
            "lag_6":  history[-6],
            "lag_12": history[-12],
            "month":  future_month,
            "t":      future_t,
        }
        x = pd.DataFrame([feat])
        yhat = model.predict(x)[0]
        forecasts.append(yhat)
        history.append(yhat)
    return np.array(forecasts)


# Forecast each leaf for next 6 months
HORIZON = 6
leaf_forecasts = {l: forecast_leaf(revenue[l], horizon=HORIZON) for l in leaves}
leaf_df = pd.DataFrame(leaf_forecasts)
leaf_df.index = pd.date_range(revenue.index[-1] + pd.DateOffset(months=1), periods=HORIZON, freq="MS")

# Bottom-up total
bu_total = leaf_df.sum(axis=1)

# Independent top-down total forecast (often differs from bottom-up sum)
td_total = pd.Series(forecast_leaf(revenue["total"], horizon=HORIZON), index=leaf_df.index)

# Reconciliation: scale leaf forecasts so they sum to the top-down total
reconciled = leaf_df.multiply(td_total.values / bu_total.values, axis=0)

print(f"Forecast horizon: {HORIZON} months ({leaf_df.index[0]:%Y-%m} → {leaf_df.index[-1]:%Y-%m})")
print(f"\nBottom-up total sum:        {bu_total.sum():>14,.0f}")
print(f"Top-down total forecast:    {td_total.sum():>14,.0f}")
print(f"Reconciled bottom-up sum:   {reconciled.sum(axis=1).sum():>14,.0f}")
print(f"\nReconciled leaf forecasts (first 3 months):")
print(reconciled.head(3).round(0))


---
## Section 2 — Probabilistic / Quantile Forecasting

A point forecast is what finance gets from any model. A **distribution** is what
they need to do scenario planning, set budgets, and stress-test.

**Quantile regression** trains a separate model per quantile (P10, P50, P90)
using pinball loss. Cheap and interpretable. We use sklearn's
`GradientBoostingRegressor(loss="quantile", alpha=...)`.

### Why these specific quantiles

- **P10**: pessimistic / "if everything goes wrong" — used for **expense planning**
- **P50**: median — used for the official forecast
- **P90**: optimistic / "stretch" — used for **upside scenarios**, hiring plans

A common ServiceTitan-flavor anti-pattern: using *mean* forecasts for revenue
plans when revenue is right-skewed. Always use median (P50) for skewed targets.


In [ ]:
def quantile_forecast(series: pd.Series, horizon: int = 6, q: list = [0.1, 0.5, 0.9]):
    df = make_features(series).dropna()
    X, y = df.drop(columns="y"), df["y"]
    models = {}
    for alpha in q:
        m = GradientBoostingRegressor(loss="quantile", alpha=alpha, n_estimators=100,
                                       max_depth=3, random_state=0)
        m.fit(X, y)
        models[alpha] = m

    history = list(series.values)
    out = {alpha: [] for alpha in q}
    for h in range(horizon):
        feat = {
            "lag_1": history[-1], "lag_2": history[-2], "lag_3": history[-3],
            "lag_6": history[-6], "lag_12": history[-12],
            "month": ((series.index[-1].month + h) % 12) + 1,
            "t": len(history),
        }
        x = pd.DataFrame([feat])
        for alpha, m in models.items():
            out[alpha].append(m.predict(x)[0])
        # advance using the median for autoregressive feedback
        history.append(out[0.5][-1])
    return out


qf = quantile_forecast(revenue["total"], horizon=HORIZON)
qf_df = pd.DataFrame(qf, index=leaf_df.index)
qf_df.columns = ["P10", "P50", "P90"]
qf_df["width_pct"] = (qf_df["P90"] - qf_df["P10"]) / qf_df["P50"] * 100

print("Total revenue quantile forecast:")
print(qf_df.round(0))
print(f"\nMean prediction interval width: {qf_df['width_pct'].mean():.1f}% of P50")


In [ ]:
# ── Visualize quantile forecast ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
hist_idx = revenue.index[-12:]
ax.plot(hist_idx, revenue["total"].iloc[-12:], "k.-", label="actual")
ax.plot(qf_df.index, qf_df["P50"], "b.-", label="P50 forecast")
ax.fill_between(qf_df.index, qf_df["P10"], qf_df["P90"], alpha=0.2, label="P10-P90 band")
ax.set_title("Total Revenue: 12mo history + 6mo quantile forecast")
ax.set_ylabel("Revenue ($)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.savefig("/tmp/q_forecast.png", dpi=80)
plt.close()
print("Saved /tmp/q_forecast.png")


---
## Section 3 — Anomaly Detection in Financial Metrics

Finance hates surprises. The pipeline should flag:
- Step changes (acquisition, big customer churn)
- Spikes (one-time billing event)
- Drift (slow margin compression)

We use two complementary detectors:

| Detector | Catches | Limitation |
|----------|---------|-----------|
| **Rolling z-score** | Sudden spikes / drops vs recent history | Misses gradual drift |
| **Isolation Forest** | Outliers in multi-feature space | Needs enough data, less interpretable |

In production, run both, alert on agreement.


In [ ]:
def rolling_zscore(series: pd.Series, window: int = 12, threshold: float = 2.5) -> pd.DataFrame:
    """Flag points more than `threshold` std-devs from rolling mean."""
    rolling_mean = series.shift(1).rolling(window).mean()
    rolling_std = series.shift(1).rolling(window).std()
    z = (series - rolling_mean) / rolling_std
    return pd.DataFrame({
        "value": series, "rolling_mean": rolling_mean, "rolling_std": rolling_std,
        "z": z, "anomaly": z.abs() > threshold,
    })


# Inject 2 anomalies into total revenue
anomalous = revenue["total"].copy()
anomalous.iloc[18] *= 1.4   # spike
anomalous.iloc[28] *= 0.6   # drop

z_results = rolling_zscore(anomalous, window=12, threshold=2.5)
anomalies = z_results[z_results["anomaly"]]
print(f"Rolling z-score caught {len(anomalies)} anomalies at indices:")
for idx, row in anomalies.iterrows():
    print(f"  {idx:%Y-%m}  value={row['value']:>14,.0f}  z={row['z']:+.2f}")


In [ ]:
# ── Isolation Forest on multivariate features ────────────────────────────
# Combine: leaf-level revenues + their MoM growth + segment shares
features = revenue[leaves].copy()
for l in leaves:
    features[f"{l}_growth"] = revenue[l].pct_change()
features = features.dropna()

iso = IsolationForest(contamination=0.1, random_state=0)
features["anomaly_score"] = iso.fit_predict(features)
features["is_anomaly"] = features["anomaly_score"] == -1

n_anomalies = features["is_anomaly"].sum()
print(f"Isolation Forest flagged {n_anomalies} months as anomalous ({n_anomalies/len(features):.0%})")
print("\nFlagged months:")
print(features[features["is_anomaly"]].index.strftime("%Y-%m").tolist())


---
## Section 4 — Resource-Constrained Budget Optimization

**Problem (real ST FP&A flavor)**: allocate next year's $20M marketing budget
across channels to maximize expected revenue, subject to constraints:

- Total spend ≤ $20M
- Each channel has a minimum (contractual) and maximum (saturation) spend
- Diminishing returns on each channel (log-utility per dollar)

This is a constrained nonlinear optimization. We use `scipy.optimize.minimize`
with SLSQP, the standard solver for smooth constrained problems.

### Why not linprog?

`linprog` only handles linear objectives. Diminishing returns → log utility →
nonlinear. If your utility is linear (a rare special case), `linprog` is faster
and globally optimal; we show both.


In [ ]:
# Channel definitions: (name, min_spend, max_spend, return_coef)
# revenue = coef * log(1 + spend)
CHANNELS = [
    ("paid_search",  500_000,  8_000_000, 2_500_000),
    ("display",      100_000,  3_000_000, 1_200_000),
    ("social",       200_000,  5_000_000, 1_800_000),
    ("partnerships", 300_000,  4_000_000, 2_100_000),
    ("events",       100_000,  2_000_000, 800_000),
]
TOTAL_BUDGET = 20_000_000

names = [c[0] for c in CHANNELS]
lo = np.array([c[1] for c in CHANNELS])
hi = np.array([c[2] for c in CHANNELS])
coef = np.array([c[3] for c in CHANNELS])


def neg_revenue(x):
    """Total expected revenue (negated for minimize)."""
    return -np.sum(coef * np.log1p(x))


# SLSQP setup
constraints = [
    {"type": "ineq", "fun": lambda x: TOTAL_BUDGET - x.sum()},   # x.sum() <= budget
]
bounds = list(zip(lo, hi))

# Sensible warm start: proportional to coefs, scaled to budget
x0 = coef / coef.sum() * TOTAL_BUDGET
x0 = np.clip(x0, lo, hi)
# Renormalize if clipping pushed total over budget
if x0.sum() > TOTAL_BUDGET:
    x0 = x0 * (TOTAL_BUDGET / x0.sum())

result = minimize(neg_revenue, x0, method="SLSQP", bounds=bounds, constraints=constraints,
                   options={"ftol": 1e-9})

print(f"Solver status:    {result.message}")
print(f"Optimal revenue:  ${-result.fun:,.0f}")
print(f"Total spend:      ${result.x.sum():,.0f}  (budget: ${TOTAL_BUDGET:,.0f})")
print()
print(f"{'Channel':<14} {'Min':>11} {'Max':>11} {'Optimal':>11} {'% of budget':>12}")
for n, l, h, opt in zip(names, lo, hi, result.x):
    pct = opt / TOTAL_BUDGET * 100
    print(f"{n:<14} {l:>11,.0f} {h:>11,.0f} {opt:>11,.0f} {pct:>11.1f}%")


In [ ]:
# ── Linear baseline (linprog) for comparison ──────────────────────────────
# Linear approximation: revenue = coef[i] * x[i] / hi[i]  (perfect substitution within bounds)
# This is degenerate — solution is just "max out the highest coef/cost channel"

# linprog minimizes c @ x.  We want to maximize sum(coef * x / hi), so c = -coef / hi
c_linear = -coef / hi
A_ub = np.ones((1, len(coef)))
b_ub = [TOTAL_BUDGET]
res_lp = linprog(c_linear, A_ub=A_ub, b_ub=b_ub, bounds=list(zip(lo, hi)), method="highs")

print("Linear baseline (no diminishing returns):")
print(f"  Status: {res_lp.message}")
print(f"  Allocation: {dict(zip(names, res_lp.x.round(0)))}")
print(f"  Compare to nonlinear (with diminishing returns) — the SLSQP result diversifies more.")


---
## Section 5 — Monte Carlo Scenario Simulation

Once you have a probabilistic forecast and an allocation, simulate the joint
distribution of outcomes:

```
For each of N scenarios:
    sample revenue forecast from quantile distribution
    sample cost inflation from prior
    compute resulting margin
Aggregate the N runs → distribution of margins
```

Report: P10 margin (downside), P50 (central case), P90 (upside).


In [ ]:
def simulate_margin(n_scenarios: int = 5_000, base_revenue_p50: float = 50_000_000):
    """Monte Carlo over revenue and cost uncertainty → margin distribution."""
    rng = np.random.default_rng(0)
    # revenue: log-normal centered at P50, sigma chosen so P90/P50 ≈ 1.15
    revenue = rng.lognormal(mean=np.log(base_revenue_p50), sigma=0.12, size=n_scenarios)
    # cost ratio: beta around 0.65 (35% baseline margin), inflation noise
    cost_ratio = rng.normal(0.65, 0.04, n_scenarios).clip(0.4, 0.9)
    margin = revenue * (1 - cost_ratio)
    return margin


margins = simulate_margin(n_scenarios=10_000)

print(f"Margin distribution over {len(margins):,} scenarios:")
print(f"  P10 (downside):    ${np.percentile(margins, 10):>14,.0f}")
print(f"  P50 (central):     ${np.percentile(margins, 50):>14,.0f}")
print(f"  P90 (upside):      ${np.percentile(margins, 90):>14,.0f}")
print(f"  Mean:              ${margins.mean():>14,.0f}")
print(f"  Std:               ${margins.std():>14,.0f}")
print(f"  P(margin < $15M):  {(margins < 15_000_000).mean():.1%}")


---
## Section 6 — End-to-End: Forecast → Optimize → Simulate

Tying the three pieces together. The production pattern:

```
1. Forecast: P10/P50/P90 revenue by segment (Section 2)
2. Optimize: allocate marketing budget assuming P50 revenue (Section 4)
3. Simulate: run MC under P10/P50/P90 to get margin distribution (Section 5)
4. Alert: if P10 margin < threshold, escalate to finance leadership
```

In a real ST FP&A pipeline this runs monthly, with the forecast tagged by
**vintage** so you can compare predicted-vs-actual over time. The simulation
results feed into the board deck for QBR.


In [ ]:
# ── Pipeline run ──────────────────────────────────────────────────────────
def run_pipeline():
    # Step 1: forecast next year's total revenue (median)
    total_forecast = quantile_forecast(revenue["total"], horizon=12, q=[0.1, 0.5, 0.9])
    p10 = sum(total_forecast[0.1])
    p50 = sum(total_forecast[0.5])
    p90 = sum(total_forecast[0.9])
    print(f"Step 1 — annual revenue forecast")
    print(f"  P10 / P50 / P90: ${p10:,.0f} / ${p50:,.0f} / ${p90:,.0f}")

    # Step 2: optimize marketing budget under expected revenue
    result = minimize(neg_revenue, x0, method="SLSQP", bounds=bounds, constraints=constraints)
    expected_mkt_revenue = -result.fun
    print(f"\nStep 2 — marketing optimization")
    print(f"  Expected incremental revenue from $20M budget: ${expected_mkt_revenue:,.0f}")

    # Step 3: simulate margin under base + uplift
    base_revenue_p50 = p50 + expected_mkt_revenue
    margins = simulate_margin(n_scenarios=10_000, base_revenue_p50=base_revenue_p50)
    print(f"\nStep 3 — margin Monte Carlo")
    print(f"  P10 margin: ${np.percentile(margins, 10):,.0f}")
    print(f"  P50 margin: ${np.percentile(margins, 50):,.0f}")
    print(f"  P90 margin: ${np.percentile(margins, 90):,.0f}")

    # Step 4: alert
    threshold = 15_000_000
    p_below = (margins < threshold).mean()
    if p_below > 0.20:
        print(f"\n  🚨 ALERT: {p_below:.1%} probability margin < ${threshold:,}")
    else:
        print(f"\n  ✓ Within tolerance: {p_below:.1%} chance margin < ${threshold:,}")

run_pipeline()


In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────
print("=" * 60)
print("Notebook 23 — Financial Forecasting & Budget Optimization")
print("=" * 60)
print(f"  Section 1 — hierarchical reconciliation: {len(leaves)} leaves")
print(f"  Section 2 — quantile forecasting:        P10/P50/P90 GBMs")
print(f"  Section 3 — anomaly detection:           rolling z + IsolationForest")
print(f"  Section 4 — budget optimization:         SLSQP + linprog comparison")
print(f"  Section 5 — Monte Carlo simulation:      10K scenarios")
print(f"  Section 6 — end-to-end pipeline:         executed")
print("=" * 60)
